chatbot basic

필요한 파츠
1. Face detection (자동 켜짐?)
2. STT (화자의 발언내용 문자화)
3. LLM (문자화된 발언 내용 처리)
4. TTS (처리 결과 음성재생)
5. speech - lip motion sync (음성 재생 시)

In [11]:
!pip install transformers==4.40.0 accelerate
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python #양자화 구동을 위한 Llama C++ 설치
!huggingface-cli download MLP-KTLim/llama-3-Korean-Bllossom-8B-gguf-Q4_K_M --local-dir='./' #Bllossom모델 다운로드

Fetching 6 files: 100%|███████████████████████| 6/6 [00:00<00:00, 112347.43it/s]
/home/DB/data/jylee/gradio-av-io


In [12]:
!pip install git+https://github.com/abetlen/llama-cpp-python.git
from llama_cpp import Llama
from transformers import AutoTokenizer

  Cloning https://github.com/abetlen/llama-cpp-python.git to /tmp/pip-req-build-8b2sawk6
  Running command git clone --filter=blob:none --quiet https://github.com/abetlen/llama-cpp-python.git /tmp/pip-req-build-8b2sawk6
  Resolved https://github.com/abetlen/llama-cpp-python.git to commit 077ecb6771fbd373d5507444f6e0b6e9bb7cf4e8
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


여기에 faster whisper 달아보자

In [13]:
!pip install openai-whisper
!pip install SpeechRecognition
import argparse
import os
import numpy as np
import speech_recognition as sr
import whisper
import torch

from datetime import datetime, timedelta
from queue import Queue
from time import sleep
from sys import platform

In [14]:
model_id = 'MLP-KTLim/llama-3-Korean-Bllossom-8B-gguf-Q4_K_M'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = Llama(
    model_path='./llama-3-Korean-Bllossom-8B-Q4_K_M.gguf', #다운로드받은 모델의 위치
    n_ctx=512,
    n_gpu_layers=-1        # Number of model layers to offload to GPU
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from ./llama-3-Korean-Bllossom-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = llama-3-Korean-Bllossom-8B
llama_model_loader: - kv   2:                          llama.block_count u32              = 32
llama_model_loader: - kv   3:                       llama.context_length u32              = 8192
llama_model_loader: - kv   4:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6: 

In [23]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default="medium", help="Model to use",
                        choices=["tiny", "base", "small", "medium", "large"])
    parser.add_argument("--non_english", action='store_true',
                        help="Don't use the english model.")
    parser.add_argument("--energy_threshold", default=1000,
                        help="Energy level for mic to detect.", type=int)
    parser.add_argument("--record_timeout", default=2,
                        help="How real time the recording is in seconds.", type=float)
    parser.add_argument("--phrase_timeout", default=3,
                        help="How much empty space between recordings before we "
                             "consider it a new line in the transcription.", type=float)
    if 'linux' in platform:
        parser.add_argument("--default_microphone", default='pulse',
                            help="Default microphone name for SpeechRecognition. "
                                 "Run this with 'list' to view available Microphones.", type=str)
    args = parser.parse_args()

    # The last time a recording was retrieved from the queue.
    phrase_time = None
    # Thread safe Queue for passing data from the threaded recording callback.
    data_queue = Queue()
    # We use SpeechRecognizer to record our audio because it has a nice feature where it can detect when speech ends.
    recorder = sr.Recognizer()
    recorder.energy_threshold = args.energy_threshold
    # Definitely do this, dynamic energy compensation lowers the energy threshold dramatically to a point where the SpeechRecognizer never stops recording.
    recorder.dynamic_energy_threshold = False

    PROMPT = \
'''당신은 유용한 AI 어시스턴트입니다. 사용자의 질의에 대해 친절하고 정확하게 답변해야 합니다.
You are a helpful AI assistant, you'll need to answer users' queries in a friendly and accurate manner.'''

    # Important for linux users.
    # Prevents permanent application hang and crash by using the wrong Microphone
    if 'linux' in platform:
        mic_name = args.default_microphone
        if not mic_name or mic_name == 'list':
            print("Available microphone devices are: ")
            for index, name in enumerate(sr.Microphone.list_microphone_names()):
                print(f"Microphone with name \"{name}\" found")
            return
        else:
            for index, name in enumerate(sr.Microphone.list_microphone_names()):
                if mic_name in name:
                    source = sr.Microphone(sample_rate=16000, device_index=index)
                    break
    else:
        source = sr.Microphone(sample_rate=16000)

    # Load / Download model
    model = args.model
    if args.model != "large" and not args.non_english:
        model = model + ".en"
    audio_model = whisper.load_model(model)

    record_timeout = args.record_timeout
    phrase_timeout = args.phrase_timeout

    transcription = ['']

    with source:
        recorder.adjust_for_ambient_noise(source)

    def record_callback(_, audio:sr.AudioData) -> None:
        """
        Threaded callback function to receive audio data when recordings finish.
        audio: An AudioData containing the recorded bytes.
        """
        # Grab the raw bytes and push it into the thread safe queue.
        data = audio.get_raw_data()
        data_queue.put(data)

    # Create a background thread that will pass us raw audio bytes.
    # We could do this manually but SpeechRecognizer provides a nice helper.
    recorder.listen_in_background(source, record_callback, phrase_time_limit=record_timeout)

    # Cue the user that we're ready to go.
    print("Model loaded.\n")

    while True:
        try:
            now = datetime.utcnow()
            # Pull raw recorded audio from the queue.
            if not data_queue.empty():
                phrase_complete = False
                # If enough time has passed between recordings, consider the phrase complete.
                # Clear the current working audio buffer to start over with the new data.
                if phrase_time and now - phrase_time > timedelta(seconds=phrase_timeout):
                    phrase_complete = True
                # This is the last time we received new audio data from the queue.
                phrase_time = now
                
                # Combine audio data from queue
                audio_data = b''.join(data_queue.queue)
                data_queue.queue.clear()
                
                # Convert in-ram buffer to something the model can use directly without needing a temp file.
                # Convert data from 16 bit wide integers to floating point with a width of 32 bits.
                # Clamp the audio stream frequency to a PCM wavelength compatible default of 32768hz max.
                audio_np = np.frombuffer(audio_data, dtype=np.int16).astype(np.float32) / 32768.0

                # Read the transcription.
                result = audio_model.transcribe(audio_np, fp16=torch.cuda.is_available())
                text = result['text'].strip()

                # If we detected a pause between recordings, add a new item to our transcription.
                # Otherwise edit the existing one.
                if phrase_complete:
                    transcription.append(text)
                else:
                    transcription[-1] = text

                # Clear the console to reprint the updated transcription.
                os.system('cls' if os.name=='nt' else 'clear')
                for line in transcription:
                    print(line)
                # Flush stdout.
                print('', end='', flush=True)

                instruction = text

                messages = [
                    {"role": "system", "content": f"{PROMPT}"},
                    {"role": "user", "content": f"{instruction}"}
                    ]

                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize = False,
                    add_generation_prompt=True
                )

                generation_kwargs = {
                    "max_tokens":512,
                    "stop":["<|eot_id|>"],
                    "top_p":0.9,
                    "temperature":0.6,
                    "echo":True, # Echo the prompt in the output
                }

                resonse_msg = model(prompt, **generation_kwargs)
                print(resonse_msg['choices'][0]['text'][len(prompt):])

            else:
                # Infinite loops are bad for processors, must sleep.
                sleep(0.25)
        except KeyboardInterrupt:
            break

    print("\n\nTranscription:")
    for line in transcription:
        print(line)

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--model {tiny,base,small,medium,large}]
                             [--non_english]
                             [--energy_threshold ENERGY_THRESHOLD]
                             [--record_timeout RECORD_TIMEOUT]
                             [--phrase_timeout PHRASE_TIMEOUT]
                             [--default_microphone DEFAULT_MICROPHONE]
ipykernel_launcher.py: error: unrecognized arguments: --f=/home/jylee/.local/share/jupyter/runtime/kernel-v2-718455pFpgzvnjW2rr.json


SystemExit: 2

In [21]:
PROMPT = \
'''당신은 유용한 AI 어시스턴트입니다. 사용자의 질의에 대해 친절하고 정확하게 답변해야 합니다.
You are a helpful AI assistant, you'll need to answer users' queries in a friendly and accurate manner.'''

while True:

    instruction = input("inst : (Press Q to quit)")

    if instruction.strip().upper() == "Q":
        print("Exiting")
        break

    messages = [
        {"role": "system", "content": f"{PROMPT}"},
        {"role": "user", "content": f"{instruction}"}
        ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt=True
    )

    generation_kwargs = {
        "max_tokens":512,
        "stop":["<|eot_id|>"],
        "top_p":0.9,
        "temperature":0.6,
        "echo":True, # Echo the prompt in the output
    }

    resonse_msg = model(prompt, **generation_kwargs)
    print(resonse_msg['choices'][0]['text'][len(prompt):])

if __name__ == "__main__":
    main()

Exiting


usage: ipykernel_launcher.py [-h] [--model {tiny,base,small,medium,large}]
                             [--non_english]
                             [--energy_threshold ENERGY_THRESHOLD]
                             [--record_timeout RECORD_TIMEOUT]
                             [--phrase_timeout PHRASE_TIMEOUT]
                             [--default_microphone DEFAULT_MICROPHONE]
ipykernel_launcher.py: error: unrecognized arguments: --f=/home/jylee/.local/share/jupyter/runtime/kernel-v2-718455pFpgzvnjW2rr.json


SystemExit: 2

In [19]:
if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--model {tiny,base,small,medium,large}]
                             [--non_english]
                             [--energy_threshold ENERGY_THRESHOLD]
                             [--record_timeout RECORD_TIMEOUT]
                             [--phrase_timeout PHRASE_TIMEOUT]
                             [--default_microphone DEFAULT_MICROPHONE]
ipykernel_launcher.py: error: unrecognized arguments: --f=/home/jylee/.local/share/jupyter/runtime/kernel-v2-718455pFpgzvnjW2rr.json


SystemExit: 2